<a href="https://colab.research.google.com/github/Calvin-Gacheru/Association-Rules/blob/main/Recommender_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1) Data preparation

The dataset is grouped by `Member_number` and `Date` so each row represents a single chechout transaction. The data is then transformed into a one-hot encoded matrix for the Apriori algorithm


In [3]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv("Groceries_dataset.csv")
df.head()

,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [4]:
# Group items into individual transactions
transactions = df.groupby(['Member_number', 'Date'])['itemDescription'].apply(list).values.tolist()

# One-hot encode the transactions
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)
df_encoded.head()

,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,beverages,bottled beer,bottled water,brandy,brown bread,butter,butter milk,cake bar,candles,candy,canned beer,canned fish,canned fruit,canned vegetables,cat food,cereals,chewing gum,chicken,chocolate,chocolate marshmallow,citrus fruit,cleaner,cling film/bags,cocoa drinks,coffee,condensed milk,cooking chocolate,cookware,cream,cream cheese,...,salt,salty snack,sauces,sausage,seasonal products,semi-finished bread,shopping bags,skin care,sliced cheese,snack products,soap,soda,soft cheese,softener,soups,sparkling wine,specialty bar,specialty cheese,specialty chocolate,specialty fat,specialty vegetables,spices,spread cheese,sugar,sweet spreads,syrup,tea,tidbits,toilet cleaner,tropical fruit,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


2) Identify Frequent Itemsets

The Apriori algorithm extracts itemsets meeting a minimum support threshold. Support metrics measures how frequent an itemset appears in the dataset. A support on 0.01/1% is selected. Grocery data is highly sparse; this threshold ensures we capture itemsets appearing in at least ~150 of the 14,963 total transactions.

In [6]:
# Identify frequent itemsets with minimum support of 1%
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values(by="support", ascending=False)
frequent_itemsets.head(10)

,support,itemsets
62,0.157923,(whole milk)
40,0.122101,(other vegetables)
46,0.110005,(rolls/buns)
52,0.097106,(soda)
63,0.085879,(yogurt)
47,0.069572,(root vegetables)
57,0.067767,(tropical fruit)
5,0.060683,(bottled water)
49,0.060349,(sausage)
15,0.053131,(citrus fruit)


The support for `whole milk` is approximately 0.1579. This means that `whole milk` is present in about `15.79%` of all transactions in the dataset.

3) Generate Association Rules

Association rules are generated from the frequent itemsets using a minimum confidence on 0.05.
This filters relationships to show only rules where purchasing the antecedent leads to purchasing the consequent at least 5% of the time

- **Antecedent(The "if")**: The item/group of items a customer already has in their basket.

- **Consequent(The "then")**: The item a customer is likely to add to their basket next.

- **Support**: How often an item or specific combination appears across all transactions. Example: If 5 out of 100 store transactions include both brea and butter, the support is 5%.

- **Confidence**: How likely it is that the consequent is purchased given that the antecedent is already in the basket. Example: If 10 people bought bread, and 6 of those people also bough butter, the confidence is 60%.

- **Lift**: How much more likely the items are bought together compared to random chance. Example: A lift of 2 means a customer buying bread is twice as likely to buy butter compared to the average shopper. A lift greater than 1 shows a strong, non-random relationship.


In [7]:
# Generate rules with a min confidence on 5%
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.05)
rules = rules.sort_values(by="lift", ascending=False)
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

,antecedents,consequents,support,confidence,lift
6,(yogurt),(whole milk),0.011161,0.129961,0.822940
7,(whole milk),(yogurt),0.011161,0.070673,0.822940
2,(rolls/buns),(whole milk),0.013968,0.126974,0.804028
3,(whole milk),(rolls/buns),0.013968,0.088447,0.804028
8,(other vegetables),(rolls/buns),0.010559,0.086481,0.786154
9,(rolls/buns),(other vegetables),0.010559,0.095990,0.786154
0,(other vegetables),(whole milk),0.014837,0.121511,0.769430
1,(whole milk),(other vegetables),0.014837,0.093948,0.769430
5,(whole milk),(soda),0.011629,0.073635,0.758296
4,(soda),(whole milk),0.011629,0.119752,0.758296


4) Observations and Conclusions

- Dominant Items: "Whole milk" is the highest-frequency item. Customers buying yogurt, rolls/buns, or other vegetables have the highest likelihood of adding whole milk to their basket.

- Support Levels: Support for item pairs is low (between 1.0% and 1.5%). This is standard for grocery data due to large product catalogs and relatively small individual transaction sizes.

- Confidence & Lift: The strongest rules yield a confidence of roughly 12-13% (e.g., yogurt -> whole milk). While numerically low, these relationships provide a positive lift over random chance.

- Conclusion: Core staple goods drive multi-item purchases. Store layout strategies should leverage these relationships by positioning mid-frequency items (like pastries or yogurt) near high-frequency anchors (like milk) to maximize cross-selling.   